# Project 08: Deep Metric Learning & Biometric Facial Recognition
## VisionAttend AI — Mathematical Foundations & Metric Evaluation

**Author:** Ali Khan  
**Portfolio:** Deep Learning & Computer Vision Progression (Project 08)  

---
### Overview
Traditional image classification maps inputs into fixed predefined class logits using Softmax cross-entropy. However, in **real-world enterprise biometric access**, new employees arrive and leave constantly. Retraining a 20-million parameter convolutional network every time a new team member joins is completely unviable.

Instead, modern facial recognition is formulated as **Deep Metric Learning** (Open-Set Verification):
The deep neural network acts as an embedding function $f_\theta: \mathcal{X} \rightarrow \mathbb{R}^{512}$ mapping high-dimensional pixel matrices onto a unit hypersphere $\mathcal{S}^{511}$ where geometric distance directly reflects biometric identity.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from facenet_pytorch import InceptionResnetV1

# Verify PyTorch environment
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Executing on hardware accelerator: {device.upper()}')

# Load pretrained InceptionResnetV1 (FaceNet) trained on VGGFace2
model = InceptionResnetV1(pretrained='vggface2').eval().to(device)
param_count = sum(p.numel() for p in model.parameters())
print(f'Successfully instantiated InceptionResnetV1 model. Parameters: {param_count:,}')

---
## 1. Mathematical Relationship: Cosine Similarity vs. Euclidean Distance

For two unit-length vectors $u, v \in \mathbb{R}^D$ where $\|u\|_2 = \|v\|_2 = 1$:

$$\begin{aligned}
\|u - v\|_2^2 &= (u - v) \cdot (u - v) \\
&= \|u\|_2^2 + \|v\|_2^2 - 2(u \cdot v) \\
&= 1 + 1 - 2 \cos(\theta) \\
&= 2 - 2 \cdot \text{Sim}_{\text{cosine}}(u, v)
\end{aligned}$$

Therefore:
$$\text{Sim}_{\text{cosine}}(u, v) = 1 - \frac{1}{2}\|u - v\|_2^2$$

This proves that maximizing cosine similarity on $L_2$-normalized embeddings is mathematically identical to minimizing Euclidean distance!

In [ ]:
def cosine_similarity(u, v):
    return float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v)))

def euclidean_distance(u, v):
    return float(np.linalg.norm(u - v))

# Generate synthetic unit vectors with varying degrees of correlation
np.random.seed(42)
base = np.random.randn(512)
base /= np.linalg.norm(base)

sims = []
euclids = []
for alpha in np.linspace(0, np.pi, 100):
    noise = np.random.randn(512)
    noise -= np.dot(noise, base) * base
    noise /= np.linalg.norm(noise)
    rotated = np.cos(alpha) * base + np.sin(alpha) * noise
    sims.append(cosine_similarity(base, rotated))
    euclids.append(euclidean_distance(base, rotated))

print('Sample verification points:')
print(f'Angle 0°   -> Cosine: {sims[0]:.4f}, Euclidean: {euclids[0]:.4f}')
print(f'Angle 60°  -> Cosine: {sims[33]:.4f}, Euclidean: {euclids[33]:.4f}')
print(f'Angle 180° -> Cosine: {sims[-1]:.4f}, Euclidean: {euclids[-1]:.4f}')

---
## 2. Multi-Shot Centroid Aggregation Formulation

In corporate deployments, a single face capture can suffer from subtle shadows or head rotation. During employee enrollment in **VisionAttend AI**, we sample $K$ poses (frontal, left 15°, right 15°) with embeddings $\{e_1, e_2, \dots, e_K\}$.

The optimal centroid vector $\mu$ minimizing mean squared angular error is computed via vector sum normalization:
$$\mu = \frac{\sum_{k=1}^K e_k}{\left\|\sum_{k=1}^K e_k\right\|_2}$$

This ensures maximum robustness against varying camera angles during live check-in!

In [ ]:
# Simulate 3-shot enrollment embeddings
shot1 = base + np.random.normal(0, 0.05, 512) # Frontal
shot2 = base + np.random.normal(0, 0.08, 512) # Left angle
shot3 = base + np.random.normal(0, 0.08, 512) # Right angle

shots = [s / np.linalg.norm(s) for s in [shot1, shot2, shot3]]
centroid = np.sum(shots, axis=0)
centroid /= np.linalg.norm(centroid)

print('Shot 1 similarity to Centroid:', round(cosine_similarity(shots[0], centroid), 4))
print('Shot 2 similarity to Centroid:', round(cosine_similarity(shots[1], centroid), 4))
print('Shot 3 similarity to Centroid:', round(cosine_similarity(shots[2], centroid), 4))

---
## 3. Threshold Calibration: ROC Curve & Equal Error Rate (EER)

In biometric verification:
- **False Acceptance Rate (FAR):** Fraction of impostor pairs falsely accepted ($> \tau$).
- **False Rejection Rate (FRR):** Fraction of genuine pairs falsely rejected ($< \tau$).

The **Equal Error Rate (EER)** is the operational operating point where $\text{FAR}(\tau) = \text{FRR}(\tau)$.
For FaceNet on VGGFace2, $\tau_{\text{cosine}} \approx 0.65$ yields a high-security operational trade-off with $\text{FAR} < 0.1\%$.